# Colab: groundwater PINN

**Before you start:** choose **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
Keep this browser tab open while it runs. Colab stops if the tab sits idle for too long.

**One-time setup:**
1. On the laptop, run `python scripts/pack_for_colab.py`. This makes `pinn_code.zip`.
2. Upload `pinn_code.zip` to Google Drive in a folder called **`pinn`** (`MyDrive/pinn/pinn_code.zip`).

**What lives where on Drive (`MyDrive/pinn/`):**

| Path | What |
|---|---|
| `pinn_code.zip` | the code (re-upload it whenever the code changes) |
| `runs/` | every run's config, metrics, history and model. Written straight to Drive, so nothing is lost if Colab disconnects |
| `runs/logs/` | the training printout of each batch of runs |
| `m2_results.zip` | small bundle of results (no model files) to download and give back to Claude |

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/pinn"
REPO_URL = ""      # leave empty: the code comes from pinn_code.zip

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/runs/logs", exist_ok=True)

In [ ]:
import shutil
import zipfile

CODE = "/content/pinn"
if REPO_URL:
    if os.path.exists(CODE):
        !git -C {CODE} pull
    else:
        !git clone {REPO_URL} {CODE}
else:
    shutil.rmtree(CODE, ignore_errors=True)
    with zipfile.ZipFile(f"{DRIVE_ROOT}/pinn_code.zip") as z:
        z.extractall(CODE)
%cd {CODE}

In [ ]:
# PyTorch is already installed on Colab; this adds everything else (about 1 minute).
!pip install -q -r requirements.txt
!pip install -q -e .

In [ ]:
# Run outputs go to Drive; data (the simulated aquifers) stays on this machine's fast disk.
os.environ["PINN_RUNS_DIR"] = f"{DRIVE_ROOT}/runs"
!python scripts/check_env.py
!python -m pytest -q -m "not data"

## Build the simulated aquifers (about 30 seconds)

In [ ]:
!python scripts/make_synthetic.py

## M2 run matrix

Each run trains one PINN on a simulated aquifer and scores it against the known truth. Expect roughly
15–25 minutes per run on a T4.

| Run | What it tests |
|---|---|
| run1 --seed 1 | the gate again with a different random start and well layout (was the pass luck?) |
| run2 | the brief's original pumping network (can use x, y, t freely) |
| run3 | realistic: 30 wells, 4 readings a year |
| run4 | + 0.3 m measurement noise |
| run6 | recharge told 20% too low |
| run9 | the deliberately unfair case D |

The extra runs in the next cell (run5, run7, run8, run10) are optional. Run them only if time allows.

In [ ]:
!python -u scripts/run_m2.py run1 --seed 1 --tag seed1 2>&1 | tee -a {DRIVE_ROOT}/runs/logs/m2_matrix.log
!python -u scripts/run_m2.py run2 run3 run4 run6 run9 2>&1 | tee -a {DRIVE_ROOT}/runs/logs/m2_matrix.log

In [ ]:
# Optional extra runs
!python -u scripts/run_m2.py run5 run7 run8 run10 2>&1 | tee -a {DRIVE_ROOT}/runs/logs/m2_matrix.log

## Bundle the results for Claude
This makes `MyDrive/pinn/m2_results.zip`, a few MB with no model files. Download it and put it in
`C:\Users\HP\pinn\` on the laptop, then tell Claude.

In [ ]:
import glob

out = f"{DRIVE_ROOT}/m2_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for path in glob.glob(f"{DRIVE_ROOT}/runs/**/*", recursive=True):
        if os.path.isfile(path) and not path.endswith(".pt"):
            z.write(path, os.path.relpath(path, f"{DRIVE_ROOT}/runs"))
print("wrote", out, f"({os.path.getsize(out) / 1e6:.1f} MB)")